# Example 6 Only: Detailed Side-by-Side Comparison

This notebook is a self-contained reproduction of **Example 6** from `rank1_qk_experiment.ipynb`.

**Target rule:**

*If the token at position 0 is $t_3$ and the token at position 2 is $t_1$, predict $t_2$. Otherwise predict $t_4$.*

The notebook compares two input sequences at every stage of the forward pass:

- Sequence A (true branch): `[3, 0, 1, 0]`
- Sequence B (contrast case): `[0, 0, 1, 0]`

For both sequences, we show:

1. Input embeddings `X`
2. Query and key scalars `Q`, `K`
3. Scores `S = alpha * (Q K^T)` and attention `P = softmax(S)`
4. Value matrix `V_mat = X W_V`
5. Attention output `H = P V_mat`
6. Final-row vector `h_last`
7. Logits `z = h_last W_O` and probabilities `p_out`

Each step includes side-by-side prints and explicit contribution breakdowns.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

np.set_printoptions(suppress=True, linewidth=140, precision=6)
torch.set_printoptions(precision=6, sci_mode=False, linewidth=140)

# Core dimensions
V = 5
L = 4
d = V + L
d_k = 1
d_v = 2

# Example 6 scales
alpha6 = 6.0
gamma6 = 5.0

# Two sequences for side-by-side comparison
seq_A = [3, 0, 0, 0]  # Example 6 true-branch reference
seq_B = [0, 0, 1, 0]  # Same as A except first token changed from t3 -> t0

print('Dimensions:')
print(f'  V={V}, L={L}, d={d}, d_k={d_k}, d_v={d_v}')
print('Scales:')
print(f'  alpha6={alpha6}, gamma6={gamma6}')
print('Sequences:')
print(f'  A = {seq_A}')
print(f'  B = {seq_B}')

In [ ]:
def get_input_matrix(token_indices, V=5, L=4):
    tokens = torch.tensor(token_indices)
    content = F.one_hot(tokens, num_classes=V).float()
    position = F.one_hot(torch.arange(L), num_classes=L).float()
    return torch.cat([content, position], dim=-1)

def labels_for_dims(V=5, L=4):
    return [f't{i}' for i in range(V)] + [f'p{i}' for i in range(L)]

def print_side_by_side(name, A, B, fmt='{:8.4f}'):
    A_np = np.array(A, dtype=float)
    B_np = np.array(B, dtype=float)
    D_np = A_np - B_np

    print(f'\n{name}')
    print('-' * max(36, len(name)))

    if A_np.ndim == 1:
        print('  index |      A |      B |    A-B')
        for i in range(A_np.shape[0]):
            print(f'  {i:5d} | {fmt.format(A_np[i])} | {fmt.format(B_np[i])} | {fmt.format(D_np[i])}')
    elif A_np.ndim == 2:
        print('  row,col |      A |      B |    A-B')
        for i in range(A_np.shape[0]):
            for j in range(A_np.shape[1]):
                print(f'  ({i:1d},{j:1d})  | {fmt.format(A_np[i, j])} | {fmt.format(B_np[i, j])} | {fmt.format(D_np[i, j])}')
    else:
        print('Unsupported rank for side-by-side printer.')

def softmax_row_np(x):
    x = np.array(x, dtype=float)
    m = np.max(x)
    ex = np.exp(x - m)
    return ex / ex.sum()

dim_labels = labels_for_dims(V, L)
tok_labels = [f't{i}' for i in range(V)]

In [ ]:
# Build Example 6 parameter matrices exactly as in the original notebook
W_Q6 = torch.nn.Linear(d, d_k, bias=False)
W_K6 = torch.nn.Linear(d, d_k, bias=False)

with torch.no_grad():
    W_Q6.weight.zero_()
    W_Q6.weight[0, V + 3] = 1.0

    W_K6.weight.zero_()
    W_K6.weight[0, 3] = -4.0
    W_K6.weight[0, 1] = 4.0
    W_K6.weight[0, V + 0] = 4.0
    W_K6.weight[0, V + 1] = -6.0
    W_K6.weight[0, V + 2] = -1.0
    W_K6.weight[0, V + 3] = -6.0

W_V6 = torch.zeros(d, d_v)
W_V6[1, 0] = 1.0
W_V6[V + 0, 0] = -1.0
W_V6[:, 1] = 0.5

W_O6 = torch.zeros(d_v, V)
W_O6[0, 2] = gamma6
W_O6[0, 4] = -gamma6
W_O6[1, 4] = gamma6

print('W_Q6 (shape d_k x d):')
print(W_Q6.weight.detach().numpy())
print('\nW_K6 (shape d_k x d):')
print(W_K6.weight.detach().numpy())
print('\nW_V6 (shape d x d_v):')
print(W_V6.detach().numpy())
print('\nW_O6 (shape d_v x V):')
print(W_O6.detach().numpy())

In [ ]:
X_A = get_input_matrix(seq_A, V=V, L=L)
X_B = get_input_matrix(seq_B, V=V, L=L)

print('X_A (A sequence embeddings):')
print(X_A.detach().numpy())
print('\nX_B (B sequence embeddings):')
print(X_B.detach().numpy())

print_side_by_side('Input matrix X entries', X_A.detach().numpy(), X_B.detach().numpy())

print('\nDimension labels (columns of X):')
for i, lab in enumerate(dim_labels):
    print(f'  {i}: {lab}')

In [ ]:
# Q and K
Q_A = W_Q6(X_A)
K_A = W_K6(X_A)

Q_B = W_Q6(X_B)
K_B = W_K6(X_B)

print('Q_A (L x 1):', Q_A.detach().numpy().flatten())
print('Q_B (L x 1):', Q_B.detach().numpy().flatten())
print_side_by_side('Q scalars by position', Q_A.detach().numpy().flatten(), Q_B.detach().numpy().flatten())

print('\nK_A (L x 1):', K_A.detach().numpy().flatten())
print('K_B (L x 1):', K_B.detach().numpy().flatten())
print_side_by_side('K scalars by position', K_A.detach().numpy().flatten(), K_B.detach().numpy().flatten())

print('\nDetailed K decomposition by position (A then B):')
for name, X_, seq_ in [('A', X_A, seq_A), ('B', X_B, seq_B)]:
    print(f'\nSequence {name} = {seq_}')
    for i in range(L):
        token = seq_[i]
        k_val = K_A[i, 0].item() if name == 'A' else K_B[i, 0].item()
        pos_term = [4.0, -6.0, -1.0, -6.0][i]
        t3_term = -4.0 if token == 3 else 0.0
        t1_term = 4.0 if token == 1 else 0.0
        print(f'  pos{i}: token=t{token}, K = pos_term({pos_term:+.1f}) + t3_term({t3_term:+.1f}) + t1_term({t1_term:+.1f}) = {k_val:+.1f}')

In [ ]:
# Scores and attention
S_A = alpha6 * (Q_A @ K_A.T)
P_A = F.softmax(S_A, dim=-1)

S_B = alpha6 * (Q_B @ K_B.T)
P_B = F.softmax(S_B, dim=-1)

print('S_A = alpha * (Q_A @ K_A^T):')
print(S_A.detach().numpy())
print('\nS_B = alpha * (Q_B @ K_B^T):')
print(S_B.detach().numpy())
print_side_by_side('Score matrix S entries', S_A.detach().numpy(), S_B.detach().numpy())

print('\nP_A = softmax(S_A) row-wise:')
print(P_A.detach().numpy())
print('\nP_B = softmax(S_B) row-wise:')
print(P_B.detach().numpy())
print_side_by_side('Attention matrix P entries', P_A.detach().numpy(), P_B.detach().numpy(), fmt='{:10.6f}')

print('\nFinal-row score and softmax decomposition (row i=3):')
for name, Q_, K_, S_, P_ in [('A', Q_A, K_A, S_A, P_A), ('B', Q_B, K_B, S_B, P_B)]:
    q_last = Q_[-1, 0].item()
    print(f'\nSequence {name}: q_last = {q_last:.1f}')
    raw = []
    for j in range(L):
        k_j = K_[j, 0].item()
        s_j = S_[-1, j].item()
        raw.append(s_j)
        print(f'  S[3,{j}] = alpha * q_last * k_{j} = {alpha6:.1f} * {q_last:.1f} * {k_j:+.1f} = {s_j:+.1f}')
    p_manual = softmax_row_np(raw)
    p_model = P_[-1].detach().numpy()
    print('  softmax(row) manual =', p_manual)
    print('  softmax(row) model  =', p_model)
    print('  max abs diff         =', np.max(np.abs(p_manual - p_model)))

In [ ]:
# Values and attention outputs
V_A = X_A @ W_V6
H_A = P_A @ V_A

V_B = X_B @ W_V6
H_B = P_B @ V_B

print('V_mat_A = X_A @ W_V6:')
print(V_A.detach().numpy())
print('\nV_mat_B = X_B @ W_V6:')
print(V_B.detach().numpy())
print_side_by_side('Value matrix V_mat entries', V_A.detach().numpy(), V_B.detach().numpy())

print('\nH_A = P_A @ V_A:')
print(H_A.detach().numpy())
print('\nH_B = P_B @ V_B:')
print(H_B.detach().numpy())
print_side_by_side('Attention output H entries', H_A.detach().numpy(), H_B.detach().numpy(), fmt='{:10.6f}')

hA = H_A[-1].detach().numpy()
hB = H_B[-1].detach().numpy()
print_side_by_side('Final-row vector h_last (2 channels)', hA, hB, fmt='{:10.6f}')

print('\nFinal-row contribution breakdown by channel:')
for name, P_last, V_ in [('A', P_A[-1].detach().numpy(), V_A.detach().numpy()), ('B', P_B[-1].detach().numpy(), V_B.detach().numpy())]:
    print(f'\nSequence {name}:')
    for c in range(d_v):
        contrib = P_last * V_[:, c]
        print(f'  Channel {c}:')
        for j in range(L):
            print(f'    j={j}: P[3,{j}] * V[{j},{c}] = {P_last[j]:.6f} * {V_[j, c]:+.6f} = {contrib[j]:+.6f}')
        print(f'    sum_j contrib = {contrib.sum():+.6f}')

In [ ]:
# Logits and output probabilities
z_A = H_A[-1].unsqueeze(0) @ W_O6
z_B = H_B[-1].unsqueeze(0) @ W_O6

p_A = F.softmax(z_A, dim=-1).squeeze()
p_B = F.softmax(z_B, dim=-1).squeeze()

print('z_A (1 x V):', z_A.detach().numpy())
print('z_B (1 x V):', z_B.detach().numpy())
print_side_by_side('Logits z by token index', z_A.detach().numpy().flatten(), z_B.detach().numpy().flatten(), fmt='{:10.6f}')

print('\np_A (output distribution):', p_A.detach().numpy())
print('p_B (output distribution):', p_B.detach().numpy())
print_side_by_side('Output probabilities p_out by token index', p_A.detach().numpy(), p_B.detach().numpy(), fmt='{:10.6f}')

pred_A = int(torch.argmax(p_A).item())
pred_B = int(torch.argmax(p_B).item())
print('\nPredictions:')
print(f'  Sequence A -> t{pred_A} (prob={p_A[pred_A].item():.6f})')
print(f'  Sequence B -> t{pred_B} (prob={p_B[pred_B].item():.6f})')

print('\nDetailed logit composition by token t: z[t] = h_last[0]*W_O[0,t] + h_last[1]*W_O[1,t]')
for name, h_last, z_ in [('A', H_A[-1].detach().numpy(), z_A.detach().numpy().flatten()), ('B', H_B[-1].detach().numpy(), z_B.detach().numpy().flatten())]:
    print(f'\nSequence {name}: h_last = {h_last}')
    for t in range(V):
        term0 = h_last[0] * W_O6[0, t].item()
        term1 = h_last[1] * W_O6[1, t].item()
        print(f'  t{t}: ({h_last[0]:+.6f}*{W_O6[0,t].item():+.1f}) + ({h_last[1]:+.6f}*{W_O6[1,t].item():+.1f}) = {term0+term1:+.6f}  (z={z_[t]:+.6f})')

In [ ]:
# Compact final side-by-side summary for all major tensors
summary = {
    'Q': (Q_A.detach().numpy().flatten(), Q_B.detach().numpy().flatten()),
    'K': (K_A.detach().numpy().flatten(), K_B.detach().numpy().flatten()),
    'S_last_row': (S_A[-1].detach().numpy(), S_B[-1].detach().numpy()),
    'P_last_row': (P_A[-1].detach().numpy(), P_B[-1].detach().numpy()),
    'h_last': (H_A[-1].detach().numpy(), H_B[-1].detach().numpy()),
    'z': (z_A.detach().numpy().flatten(), z_B.detach().numpy().flatten()),
    'p_out': (p_A.detach().numpy(), p_B.detach().numpy()),
}

print('FINAL COMPARISON SUMMARY')
print('========================')
for key, (a, b) in summary.items():
    print_side_by_side(key, a, b, fmt='{:10.6f}')

print('\nInterpretation quick check:')
print('  - A should strongly favor t2 (true branch).')
print('  - B should strongly favor t4 (first-token condition fails).')